# 5-fluorouracil Hessian, thermochemistry and IR data (Module 2) on a Colab A100

Adapted from the earlier tested Hessian notebook (L4 frequency notebook): same Skala-1.1 GPU calculator, same three-pass L-BFGS idea, same finite-difference Hessian by ASE (displacement 0.01 A, two points per coordinate). Two things are different on purpose, and you should read both:

1. **All 12 atoms are displaced** (the earlier tested cell displaced carbon atoms only). The Hessian is 73 force calculations per structure: 6 x 12 + 1.
2. **Forces are converted to true eV/A.** PySCF gradients are in Hartree/Bohr. The tested calculator multiplied them by 27.2114 and called the result eV/A, which is eV/Bohr, about 1.89 times too small. That is harmless for an optimizer with a force threshold, but in a finite-difference Hessian it scales every frequency by about 0.73. A wiring check at the start (cell 6) compares the analytic force with a finite difference of energies and **stops the notebook if they disagree**.

## What it does, per shape (T1, T3, T3B)
1. Reads the def2-SVP geometry that Module 1 left in `MyDrive/fu5_handoff/`: with `INPUT_SOURCE = "deformed6"` the file `<SHAPE>_svp_deformed6_run.xyz` from the very-deformed-start notebook (Run 3; results go to `fu5_results_run3`, never mixed with the others), with `INPUT_SOURCE = "deformed"` (default) the file `<SHAPE>_svp_deformed_run.xyz` from the deformed-start notebook, and only if its same-structure check (cell 16c) passed; with `"first_run"` the file `<SHAPE>_svp_final.xyz` from the first notebook (falls back to the MMFF start if the file is missing, with a warning).
2. Re-optimizes with Skala-1.1 / def2-TZVP in three passes.
3. Runs the finite-difference **Infrared** calculation: at every displaced geometry it stores the forces **and the dipole moment** on Google Drive (`MyDrive/fu5_results/<SHAPE>/ir_cache`). A Colab disconnect loses at most one displacement; run all again and it continues.
4. Frequencies, zero imaginary modes check, ZPE, H(298 K) and G(298 K) (ideal gas, rigid rotor, harmonic oscillator, 1 atm, symmetry number 1).
5. PBE0 single-point energies on the Skala geometries (electronic energy only; no PBE0 optimization or Hessian), for a Skala vs PBE0 comparison of the electronic gap.
6. Last cells (CPU only, can be run alone later): IR stick spectrum and broadened curve from the saved cache with an adjustable scaling factor, and the T3 - T1 comparison table (electronic energy and free energy) against the CBS-QB3 value 13.33 kcal/mol.

## Before you start
- Runtime > Change runtime type > **A100 GPU** (an L4 works but is about twice as slow at def2-TZVP).
- Run all. Expected time on an A100: about 35 to 50 minutes per shape.
- No Gemini key is needed here.

## What the IR numbers are, and are not
Harmonic, unscaled frequencies and double-harmonic intensities from Skala-1.1/def2-TZVP, gas phase. They are not anharmonic, not matrix-shifted, and not compared with any measured spectrum in this notebook.


In [ ]:
# @title 0. Install (pinned to the Skala version of the earlier tested Module 1 notebook)
SKALA_VERSION = "2026.9"   # same pin as the tested A100 notebook (gpu4pyscf 1.8.1, pyscf 2.14.0). Restart the session after changing it.
!pip install -q skala-cuda12x=={SKALA_VERSION}
!pip install -q ase


In [ ]:
# @title 1. Settings: edit this cell, then Runtime > Run all
SHAPES = ["T1"]                    # shapes to run in this session, in this order. Phase 2 (IR) needs only ["T1"].
INPUT_SOURCE = "deformed"       # setting used for the released IR result (Run 2 start). Options: "deformed" = output of the deformed-start Module 1 notebook, Run 2 (<SHAPE>_svp_deformed_run.xyz) | "deformed6" = output of the very-deformed-start notebook, Run 3 (<SHAPE>_svp_deformed6_run.xyz; results go to fu5_results_run3) | "first_run" = output of the first Module 1 notebook (<SHAPE>_svp_final.xyz)
REQUIRE_SAME_STRUCTURE = True    # with "deformed": stop unless cell 16c of the deformed-start notebook found Run 1 and Run 2 at the same structure
BACKEND = "skala"                # "skala" = real Skala-1.1 on the GPU | "standin_xtb" = CPU wiring test only (not a result)
BASIS = "def2-tzvp"
GRID_LEVEL = 3
SMALL_RHO_CUTOFF_L3 = 0.0        # Skala 2026.9 rejects pruned grids (1e-10); 0.0 keeps the full grid, as in the tested Module 1 notebook
SPHERICAL_D = True               # pure d functions (PySCF cart=False), the golden rule of the tested notebooks

# Optimizer: three passes, thresholds in TRUE eV/A (see the force-unit note above). Pass 3 is tighter than the earlier tested 0.02 (v7 units = 0.038 true) because a Hessian wants a tight minimum.
OPT_PASSES = [dict(maxstep=0.020, steps=60,  fmax=0.15, conv_tol=1e-6),
              dict(maxstep=0.010, steps=80,  fmax=0.05, conv_tol=1e-6),
              dict(maxstep=0.005, steps=120, fmax=0.01, conv_tol=1e-7)]
LBFGS_MEMORY = 20

# Hessian / IR
DELTA = 0.01                     # A, displacement (same as the tested cell)
NFREE = 2                        # central difference
HESS_CONV_TOL = 1e-8             # SCF energy tolerance for every displaced geometry (the earlier tested cell used 1e-5; tighter here so that soft modes are not noise)
TEMPERATURE_K, PRESSURE_PA = 298.15, 101325.0
SYMMETRY_NUMBER = 1              # Cs or C1 rotational symmetry number

# Wiring check (cell 6): fails the notebook if analytic force and finite-difference force differ by more than this
FD_CHECK_TOLERANCE = 0.05
ALLOW_MMFF_FALLBACK = True       # if Module 1 left no hand-off file for a shape, start from the MMFF structure (slower, more passes needed)

# Where things are stored
USE_DRIVE = True
SHAPE_NAMES = {"T1": "diketo (the drug)", "T3": "4-hydroxy-2-oxo, O-H toward C5-F", "T3B": "4-hydroxy-2-oxo, O-H toward N3"}
CBS_QB3_DG_KCAL = 13.33          # Wielinska, Nowacki, Liberek, Molecules 2019, 24, 3683, Table 1: T3 - T1, gas, 298 K
print("Shapes:", SHAPES, "| backend:", BACKEND, "| basis:", BASIS, "| grid level:", GRID_LEVEL)
if BACKEND != "skala": print("WARNING: BACKEND is not 'skala'. Numbers from this run are a wiring test only and must not be reported.")


In [ ]:
# @title 2. Google Drive, folders, GPU check
import os, json, time, platform
RES_FOLDER = "fu5_results_run3" if INPUT_SOURCE == "deformed6" else "fu5_results"   # Run 3 based results never mix with the others
HAND_DIR = "/content/fu5_handoff"; RES_ROOT = "/content/" + RES_FOLDER
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        HAND_DIR = "/content/drive/MyDrive/fu5_handoff"; RES_ROOT = "/content/drive/MyDrive/" + RES_FOLDER
        print("Google Drive mounted.")
    except Exception as e:
        print(f"Drive not mounted ({type(e).__name__}); results go to {RES_ROOT} and will vanish when Colab disconnects.")
os.makedirs(HAND_DIR, exist_ok=True); os.makedirs(RES_ROOT, exist_ok=True)
print("Hand-off folder from Module 1:", HAND_DIR, "| results folder:", RES_ROOT)

GPU_NAME = "none"
try:
    import torch
    if torch.cuda.is_available():
        GPU_NAME = torch.cuda.get_device_name(0)
        print(f"GPU: {GPU_NAME} ({torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB)")
        if "A100" not in GPU_NAME: print("Note: this notebook was planned for an A100; other GPUs work but timings differ.")
    elif BACKEND == "skala":
        raise RuntimeError("BACKEND='skala' needs a GPU. Use Runtime > Change runtime type > A100 GPU.")
except ImportError:
    if BACKEND == "skala": raise
import importlib.metadata as _imd
def _pkg(p):
    try: return _imd.version(p)
    except Exception: return None
PKGS = {p: _pkg(p) for p in ("skala", "skala-cuda12x", "gpu4pyscf-cuda12x", "pyscf", "torch", "cupy-cuda12x", "e3nn", "ase")}
json.dump({"gpu": GPU_NAME, "python": platform.python_version(), "started": time.strftime("%Y-%m-%dT%H:%M:%S"), "packages": PKGS,
           "skala_pin": SKALA_VERSION, "settings": dict(basis=BASIS, grid=GRID_LEVEL, delta=DELTA, nfree=NFREE, hess_conv_tol=HESS_CONV_TOL, small_rho_cutoff=SMALL_RHO_CUTOFF_L3)},
          open(os.path.join(RES_ROOT, "environment.json"), "w"), indent=2)
print("Packages:", PKGS)
if BACKEND == "skala" and PKGS.get("skala-cuda12x") not in (None, SKALA_VERSION): print(f"WARNING: skala-cuda12x {PKGS.get('skala-cuda12x')} is installed, the pin is {SKALA_VERSION}. Restart the session and run all again.")


In [ ]:
# @title 3. GPU memory utilities (from the earlier tested notebooks)
import gc
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
try:
    import torch
    try: torch.serialization.add_safe_globals([slice])
    except Exception: pass
    import cupy as cp
    HAVE_GPU_LIBS = True
except Exception:
    HAVE_GPU_LIBS = False
if HAVE_GPU_LIBS:
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    def hard_reset_gpu_allocator_pool(verbose=False):
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.ipc_collect()
        cp.get_default_memory_pool().free_all_blocks(); cp.get_default_pinned_memory_pool().free_all_blocks()
        cp.get_default_memory_pool().set_limit(0); cp.get_default_memory_pool().set_limit(None)
    def dynamic_step_evacuation(*a):
        torch.cuda.synchronize(); cp.cuda.Stream.null.synchronize(); gc.collect(); torch.cuda.empty_cache()
        cp.get_default_memory_pool().free_all_blocks(); cp.get_default_pinned_memory_pool().free_all_blocks()
else:
    def hard_reset_gpu_allocator_pool(verbose=False): gc.collect()
    def dynamic_step_evacuation(*a): gc.collect()


In [ ]:
# @title 4. Skala-1.1 GPU calculator (tested calculator, with TRUE eV/A forces and the dipole moment)
import numpy as np, time
from ase.calculators.calculator import Calculator, all_changes
HARTREE_TO_EV = 27.211386245988
BOHR_TO_ANG = 0.529177210903

def _to_numpy(x):
    if hasattr(x, "get"): x = x.get()
    return np.asarray(x, dtype=float)

def dipole_e_ang(cpu_mol, dm):
    """Dipole moment in e*Angstrom from the total density matrix (neutral molecule: the origin does not matter)."""
    cpu_mol.set_common_orig([0.0, 0.0, 0.0])
    ints = cpu_mol.intor_symmetric("int1e_r", comp=3)
    el = -np.einsum("xij,ji->x", ints, np.asarray(dm))
    nuc = np.einsum("i,ix->x", cpu_mol.atom_charges(), cpu_mol.atom_coords())
    return (el + nuc) * BOHR_TO_ANG

class NativeGPUSkalaCalculator(Calculator):
    """Your tested Skala-1.1 GPU calculator. Changes: (1) forces in true eV/A (Hartree/Bohr -> eV/A),
    (2) dipole moment (e*A) from the converged density, needed for IR intensities, (3) SCF convergence flag and one retry
    with level shift, (4) cart=False and small_rho_cutoff as in the tested Module 1 notebook."""
    implemented_properties = ["energy", "forces", "dipole"]

    def __init__(self, fixed_spin=0, fixed_charge=0, basis_set="def2-tzvp", grid_level=3, conv_tol=1e-6, verbose=0, max_memory=14000, **kwargs):
        super().__init__(**kwargs)
        self.fixed_spin, self.fixed_charge = fixed_spin, fixed_charge
        self.basis_set, self.grid_level, self.conv_tol, self.verbose = basis_set, grid_level, conv_tol, verbose
        self.max_memory = max_memory
        self.n_scf = 0; self.n_unconverged = 0; self.nao = None; self.last_scf = {}

    def calculate(self, atoms=None, properties=["energy"], system_changes=all_changes):
        from pyscf import gto, dft
        os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
        from skala.gpu4pyscf import SkalaKS, SkalaUKS
        super().calculate(atoms, properties, system_changes)
        t0 = time.perf_counter()
        gc.collect(); torch.cuda.empty_cache(); cp.get_default_memory_pool().free_all_blocks()
        atom_list = [[self.atoms.symbols[i], self.atoms.positions[i]] for i in range(len(self.atoms))]
        cpu_mol = gto.M(atom=atom_list, basis=self.basis_set, charge=self.fixed_charge, spin=self.fixed_spin, cart=not SPHERICAL_D, verbose=0)
        self.nao = int(cpu_mol.nao)
        gpu_mol = cpu_mol.to_gpu()
        if self.fixed_spin > 0: base_ks = SkalaUKS(gpu_mol, xc="skala-1.1"); guess = "atom"
        else:                   base_ks = SkalaKS(gpu_mol, xc="skala-1.1");  guess = "minao"
        base_ks.grids.level = self.grid_level
        if self.grid_level >= 3:
            base_ks.grids.prune = dft.gen_grid.nwchem_prune; base_ks.small_rho_cutoff = SMALL_RHO_CUTOFF_L3
        else:
            base_ks.grids.prune = dft.gen_grid.sg1_prune; base_ks.small_rho_cutoff = 1e-8
        base_ks.max_memory = self.max_memory
        base_ks.verbose = self.verbose; base_ks.conv_tol = self.conv_tol
        scf = base_ks.density_fit(auxbasis="def2-universal-jkfit"); scf.init_guess = guess
        e_ha = float(scf.kernel()); retried = False
        if not bool(getattr(scf, "converged", True)):
            retried = True
            try:
                scf.level_shift = 0.3; scf.max_cycle = 300; scf.init_guess = guess
                e_ha = float(scf.kernel())
            except Exception as ex:
                print("   SCF retry failed:", repr(ex)[:120])
        conv = bool(getattr(scf, "converged", True)); self.n_scf += 1; self.n_unconverged += int(not conv)
        self.results["energy"] = e_ha * HARTREE_TO_EV
        dm = _to_numpy(scf.make_rdm1())
        if dm.ndim == 3: dm = dm.sum(0)
        self.results["dipole"] = dipole_e_ang(cpu_mol, dm)
        if "forces" in properties:
            g = _to_numpy(scf.nuc_grad_method().kernel())            # Hartree/Bohr
            self.results["forces"] = -g * HARTREE_TO_EV / BOHR_TO_ANG   # true eV/A
        self.last_scf = dict(converged=conv, retried=retried, wall_s=round(time.perf_counter() - t0, 2))
        del gpu_mol, scf
        torch.cuda.synchronize(); cp.cuda.Stream.null.synchronize()
        gc.collect(); torch.cuda.empty_cache(); cp.get_default_memory_pool().free_all_blocks()

class StandInCalculator(Calculator):
    """CPU stand-in (GFN2-xTB) so the notebook can be tested without a GPU. Wiring test only."""
    implemented_properties = ["energy", "forces", "dipole"]
    def __init__(self, basis_set=None, conv_tol=None, **kw):
        super().__init__(**kw); from tblite.ase import TBLite
        self.tb = TBLite(method="GFN2-xTB", verbose=0); self.n_scf = 0; self.n_unconverged = 0; self.last_scf = {}; self.nao = None
    def calculate(self, atoms=None, properties=["energy"], system_changes=all_changes):
        super().calculate(atoms, properties, system_changes)
        a = self.atoms.copy(); a.calc = self.tb
        self.results["energy"] = float(a.get_potential_energy()); self.results["forces"] = np.array(a.get_forces())
        self.results["dipole"] = np.array(a.get_dipole_moment()); self.n_scf += 1; self.last_scf = dict(converged=True, retried=False, wall_s=0)

def make_calculator(basis_set, conv_tol):
    if BACKEND == "skala": return NativeGPUSkalaCalculator(fixed_spin=0, basis_set=basis_set, grid_level=GRID_LEVEL, conv_tol=conv_tol, verbose=0)
    return StandInCalculator()
print("Calculator ready. Forces are in true eV/A; dipoles in e*A.")


In [ ]:
# @title 5. Structures: Module 1 hand-off, with the MMFF start as a fallback
from ase import Atoms
from ase.io import read
import io
MMFF_STARTS = {
    "T1": "12\nMMFF start T1\nO     -2.206973    -1.429075    -0.314166\nC     -1.185701    -0.767505    -0.168743\nN     -1.223930     0.582756     0.071158\nC     -0.103547     1.353044     0.238654\nC      1.111604     0.808118     0.174000\nF      2.237260     1.527149     0.332492\nC      1.249036    -0.643875    -0.081406\nO      2.337593    -1.202881    -0.151962\nN      0.064391    -1.322184    -0.234201\nH     -2.148878     0.996792     0.120244\nH     -0.249723     2.411109     0.423549\nH      0.118868    -2.313446    -0.409619",
    "T3": "12\nMMFF start T3\nO     -2.359187    -1.662734    -0.400000\nC     -1.340352    -0.998678    -0.237131\nN     -0.076158    -1.536319    -0.284012\nC      0.908097    -0.731225    -0.096762\nO      2.172461    -1.195947    -0.130323\nC      0.825956     0.707447     0.162970\nF      1.953351     1.414188     0.338074\nC     -0.407354     1.201926     0.203208\nN     -1.467444     0.358142     0.005785\nH      2.833121    -0.500608     0.023694\nH     -0.620264     2.248017     0.385819\nH     -2.422228     0.695792     0.028679",
    "T3B": "12\nMMFF start T3B\nO     -2.359187    -1.662734    -0.400000\nC     -1.340352    -0.998678    -0.237131\nN     -0.076158    -1.536319    -0.284012\nC      0.908097    -0.731225    -0.096762\nO      2.172461    -1.195947    -0.130323\nC      0.825956     0.707447     0.162970\nF      1.953351     1.414188     0.338074\nC     -0.407354     1.201926     0.203208\nN     -1.467444     0.358142     0.005785\nH      2.217906    -2.150818    -0.303083\nH     -0.620264     2.248017     0.385819\nH     -2.422228     0.695792     0.028679",
}

def load_start(shape):
    assert INPUT_SOURCE in ("deformed", "deformed6", "first_run"), "INPUT_SOURCE must be 'deformed', 'deformed6' or 'first_run'"
    FILES = {"deformed": (f"{shape}_svp_deformed_run.xyz", f"{shape}_run1_vs_run2_same_structure.json", "Run 2", "deformed-start"),
             "deformed6": (f"{shape}_svp_deformed6_run.xyz", f"{shape}_run1_vs_run3_same_structure.json", "Run 3", "very-deformed-start"),
             "first_run": (f"{shape}_svp_final.xyz", None, "Run 1", "first")}
    fx, fj, runname, nbname = FILES[INPUT_SOURCE]; f = os.path.join(HAND_DIR, fx)
    if INPUT_SOURCE != "first_run" and not os.path.exists(f):
        raise FileNotFoundError(f"{shape}: INPUT_SOURCE is '{INPUT_SOURCE}' but {f} is not on Drive. Run the {nbname} Module 1 notebook for {shape} (and its cell 16c), or change INPUT_SOURCE for this session. Module 2 will not silently start from the MMFF structure.")
    if INPUT_SOURCE != "first_run" and REQUIRE_SAME_STRUCTURE:
        cj = os.path.join(HAND_DIR, fj)
        if not os.path.exists(cj):
            raise RuntimeError(f"{shape}: the same-structure check file is missing ({cj}). Run cell 16c of the {nbname} notebook first, or set REQUIRE_SAME_STRUCTURE = False.")
        chk = json.load(open(cj))
        if not chk.get("same_structure"):
            raise RuntimeError(f"{shape}: Run 1 and {runname} did NOT reach the same structure (energy difference {chk.get('dE_meV'):+.1f} meV, RMSD {chk.get('rmsd_all_A'):.3f} A). Stopping before Module 2. Look at cell 16c, then decide.")
        print(f"   {shape}: same-structure check passed for {runname} (energy {chk['dE_meV']:+.2f} meV, RMSD {chk['rmsd_all_A']:.4f} A)")
    if os.path.exists(f):
        a = read(f); src = f"Module 1 hand-off, {INPUT_SOURCE} ({f})"
    elif ALLOW_MMFF_FALLBACK:
        a = read(io.StringIO(MMFF_STARTS[shape].replace("\\n", "\n")), format="xyz"); src = "MMFF start (NO Module 1 hand-off file found: run Module 1 for this shape for a better start)"
        print(f"   WARNING {shape}: {src}")
    else:
        raise FileNotFoundError(f"No hand-off file {f}. Run Module 1 for shape {shape} first (or set ALLOW_MMFF_FALLBACK = True).")
    return a, src
_hf = lambda s: {"deformed": f"{s}_svp_deformed_run.xyz", "deformed6": f"{s}_svp_deformed6_run.xyz", "first_run": f"{s}_svp_final.xyz"}[INPUT_SOURCE]
print(f"Input source: {INPUT_SOURCE}. Shapes available:", {s: ("hand-off" if os.path.exists(os.path.join(HAND_DIR, _hf(s))) else "MMFF fallback") for s in SHAPES})


In [ ]:
# @title 6. Wiring check (automatic): force units, finite-difference agreement, dipole, one SCF per shape
# Analytic force on the largest-force coordinate vs central finite difference of energies (+/- 0.005 A), in def2-SVP (fast).
# This is the check that would have caught the Hartree/Bohr -> "eV/A" factor of 1.89. If it fails, nothing else runs.
def wiring_check(shape):
    a0, _ = load_start(shape); a = a0.copy()
    calc = make_calculator("def2-svp", 1e-8); a.calc = calc
    t0 = time.time(); F = a.get_forces(); E0 = a.get_potential_energy(); mu = a.get_dipole_moment()
    i, k = np.unravel_index(np.argmax(np.abs(F)), F.shape); h = 0.005
    Es = []
    for sgn in (+1, -1):
        b = a0.copy(); b.positions[i, k] += sgn * h; b.calc = make_calculator("def2-svp", 1e-8); Es.append(b.get_potential_energy())
    f_fd = -(Es[0] - Es[1]) / (2 * h); f_an = F[i, k]
    rel = abs(f_fd - f_an) / max(abs(f_an), 1e-3)
    ok = rel < FD_CHECK_TOLERANCE or abs(f_fd - f_an) < 0.01
    print(f"   {shape}: E(def2-SVP) = {E0/HARTREE_TO_EV:.6f} Eh | atom {i} axis {'xyz'[k]}: analytic {f_an:+.4f}, finite difference {f_fd:+.4f} eV/A (ratio {f_fd/f_an if abs(f_an)>1e-6 else float('nan'):.3f}) | dipole {np.linalg.norm(mu)*4.80320:.2f} D | {time.time()-t0:.0f} s -> {'OK' if ok else 'FAIL'}")
    hard_reset_gpu_allocator_pool()
    return ok
WIRING = {s: wiring_check(s) for s in SHAPES[:1]}      # one shape is enough: same code path for all three
if not all(WIRING.values()):
    raise RuntimeError("WIRING CHECK FAILED: analytic and finite-difference forces disagree. The Hessian would be wrong. Do not continue; send me this output.")
print("Wiring check passed.")


In [ ]:
# @title 7. Functions: three-pass optimization, finite-difference Infrared (forces + dipoles on Drive), thermochemistry
import shutil
from ase.optimize import LBFGS
from ase.vibrations import Infrared
from ase.thermochemistry import IdealGasThermo

KCAL = 23.060548   # kcal/mol per eV

def write_xyz(path, atoms, comment):
    open(path, "w").write(f"{len(atoms)}\n{comment}\n" + "\n".join(f"{s:2s} {p[0]:14.8f} {p[1]:14.8f} {p[2]:14.8f}" for s, p in zip(atoms.get_chemical_symbols(), atoms.positions)) + "\n")

def optimize_shape(shape, rdir):
    done = os.path.join(rdir, "opt_summary.json")
    if os.path.exists(done) and os.path.exists(os.path.join(rdir, "opt_final.xyz")):
        old = json.load(open(done))
        if old.get("input_source", INPUT_SOURCE) != INPUT_SOURCE:
            raise RuntimeError(f"{rdir} already holds results started from '{old.get('input_source')}', but INPUT_SOURCE is '{INPUT_SOURCE}'. Rename or move that folder so the two are not mixed.")
        print(f"   [skip] {shape}: optimization already finished"); return read(os.path.join(rdir, "opt_final.xyz")), old
    atoms, src = load_start(shape); write_xyz(os.path.join(rdir, "start.xyz"), atoms, f"{shape} start: {src}")
    t0 = time.time(); grads = 0; passes = []
    for p, cfg in enumerate(OPT_PASSES, 1):
        hard_reset_gpu_allocator_pool()
        atoms.calc = make_calculator(BASIS, cfg["conv_tol"])
        dyn = LBFGS(atoms, maxstep=cfg["maxstep"], memory=LBFGS_MEMORY, logfile="-"); dyn.attach(dynamic_step_evacuation)
        print(f"\n--- {shape} pass {p}/{len(OPT_PASSES)}: maxstep {cfg['maxstep']} A, fmax {cfg['fmax']} eV/A, conv_tol {cfg['conv_tol']} ---")
        conv = dyn.run(fmax=cfg["fmax"], steps=cfg["steps"]); grads += dyn.get_number_of_steps() + 1
        fm = float(np.linalg.norm(atoms.get_forces(), axis=1).max()); passes.append(dict(pass_id=p, converged=bool(conv), steps=dyn.get_number_of_steps(), fmax_eV_A=fm))
    E = float(atoms.get_potential_energy()); fm = float(np.linalg.norm(atoms.get_forces(), axis=1).max())
    summ = dict(shape=shape, input_source=INPUT_SOURCE, start=src, basis=BASIS, grid_level=GRID_LEVEL, E_eV=E, E_Eh=E / HARTREE_TO_EV, fmax_eV_A=fm, target_fmax_eV_A=OPT_PASSES[-1]["fmax"],
                converged=bool(fm <= OPT_PASSES[-1]["fmax"]), passes=passes, gradients=grads, minutes=round((time.time() - t0) / 60, 2))
    write_xyz(os.path.join(rdir, "opt_final.xyz"), atoms, f"{shape} Skala-1.1/{BASIS} optimized, E = {E/HARTREE_TO_EV:.8f} Eh, fmax {fm:.4f} eV/A")
    json.dump(summ, open(done, "w"), indent=2)
    print(f"   {shape}: optimization {'converged' if summ['converged'] else 'NOT converged'}: E = {summ['E_Eh']:.8f} Eh, fmax {fm:.4f} eV/A, {summ['minutes']} min")
    return atoms, summ

def run_infrared(shape, atoms, rdir):
    cache = os.path.join(rdir, "ir_cache")
    atoms = atoms.copy(); calc = make_calculator(BASIS, HESS_CONV_TOL); atoms.calc = calc
    if os.path.isdir(cache):                                              # a Colab disconnect can leave an empty 0-byte file for the unfinished displacement; ASE would skip it. Remove it so it is recomputed.
        for _f in os.listdir(cache):
            if _f.endswith(".json") and os.path.getsize(os.path.join(cache, _f)) == 0: os.remove(os.path.join(cache, _f)); print(f"   removed empty leftover {_f} (unfinished displacement from an interrupted run)")
    ir = Infrared(atoms, name=cache, delta=DELTA, nfree=NFREE)          # all atoms are displaced
    ntot = 6 * len(atoms) + 1 if NFREE == 2 else 12 * len(atoms) + 1
    have = len([f for f in os.listdir(cache) if f.endswith(".json")]) if os.path.isdir(cache) else 0
    print(f"   {shape}: Infrared run, {ntot} force + dipole calculations, {have} already on Drive")
    t0 = time.time(); ir.run()
    print(f"   {shape}: displaced calculations done in {(time.time()-t0)/60:.1f} min; unconverged SCF in this session: {calc.n_unconverged}")
    return ir, calc

def analyse(shape, atoms, ir, E_eV, rdir):
    n = len(atoms); freqs_c = ir.get_frequencies()                       # cm^-1, complex for imaginary
    I = np.array(ir.intensities) * ir.intensity_prefactor("km/mol")[0]    # km/mol (ASE: (D/A)^2/amu -> km/mol)
    order = np.arange(len(freqs_c)); nvib = 3 * n - 6
    vib_idx = order[-nvib:]                                               # the 6 lowest are translation/rotation residuals
    tr_res = [float(np.real(freqs_c[i]) if np.imag(freqs_c[i]) == 0 else -np.imag(freqs_c[i])) for i in order[:6]]
    vib_f = freqs_c[vib_idx]; imag = [float(np.imag(f)) for f in vib_f if np.imag(f) != 0]
    energies = ir.get_energies()[vib_idx]
    thermo = IdealGasThermo(vib_energies=energies, potentialenergy=E_eV, atoms=atoms, geometry="nonlinear", symmetrynumber=SYMMETRY_NUMBER, spin=0, ignore_imag_modes=True)
    H = thermo.get_enthalpy(TEMPERATURE_K, verbose=False); G = thermo.get_gibbs_energy(TEMPERATURE_K, PRESSURE_PA, verbose=False)
    zpe = float(np.sum(np.real(energies)) / 2)
    out = dict(shape=shape, input_source=INPUT_SOURCE, basis=BASIS, n_imaginary=len(imag), imaginary_cm1=imag, tr_residual_cm1=tr_res, lowest_real_vib_cm1=float(np.min(np.real(vib_f[np.imag(vib_f) == 0]))) if np.any(np.imag(vib_f) == 0) else None,
               E_eV=E_eV, E_Eh=E_eV / HARTREE_TO_EV, ZPE_eV=zpe, H298_eV=float(H), G298_eV=float(G), T_K=TEMPERATURE_K, P_Pa=PRESSURE_PA, symmetry_number=SYMMETRY_NUMBER,
               model="harmonic, ideal gas, rigid rotor", frequencies_cm1=[float(np.real(f)) if np.imag(f) == 0 else -float(np.imag(f)) for f in vib_f], intensities_km_mol=[float(x) for x in I[vib_idx]])
    json.dump(out, open(os.path.join(rdir, "thermo.json"), "w"), indent=2)
    with open(os.path.join(rdir, "freq_ir.csv"), "w") as f:
        f.write("mode,freq_cm1_unscaled,intensity_km_per_mol\n")
        for k, (fr, it) in enumerate(zip(out["frequencies_cm1"], out["intensities_km_mol"]), 1): f.write(f"{k},{fr:.3f},{it:.4f}\n")
    print(f"   {shape}: {len(imag)} imaginary mode(s) {imag if imag else ''} | lowest real mode {out['lowest_real_vib_cm1']:.1f} cm-1 | rigid-body residuals up to {max(abs(x) for x in tr_res):.1f} cm-1 | ZPE {zpe*KCAL:.2f} kcal/mol | G(298) = {G/HARTREE_TO_EV:.6f} Eh")
    return out
print("Functions ready.")


In [ ]:
# @title 8. Run the shapes (resumable; everything lands on Drive in fu5_results/<SHAPE>)
ALL = {}
t_all = time.time()
for shape in SHAPES:
    rdir = os.path.join(RES_ROOT, shape); os.makedirs(rdir, exist_ok=True)
    print("\n" + "=" * 90 + f"\n{shape}: {SHAPE_NAMES[shape]}\n" + "=" * 90)
    tdone = os.path.join(rdir, "thermo.json")
    atoms, osum = optimize_shape(shape, rdir)
    if osum.get("converged") is False: print(f"   WARNING {shape}: the optimization did not reach fmax {OPT_PASSES[-1]['fmax']} eV/A. The Hessian below is at an approximate minimum; read the log.")
    ir, calc = run_infrared(shape, atoms, rdir)
    ALL[shape] = analyse(shape, atoms, ir, osum["E_eV"], rdir)
    hard_reset_gpu_allocator_pool()
print(f"\nAll shapes finished in {(time.time()-t_all)/60:.1f} min. Results: {RES_ROOT}")


In [ ]:
# @title 9. PBE0 single points on the Skala geometries (electronic energy only, no PBE0 optimization, no PBE0 Hessian)
RUN_PBE0 = True
PBE0_TRY_D3BJ = True        # also try PBE0-D3(BJ) (needs the pyscf-dispersion package; skipped quietly if missing)
def pbe0_single_point(atoms, d3bj=False):
    from pyscf import gto, dft
    mol = gto.M(atom=[[s, p] for s, p in zip(atoms.get_chemical_symbols(), atoms.positions)], basis=BASIS, cart=not SPHERICAL_D, verbose=0)
    if BACKEND == "skala":
        from gpu4pyscf.dft import RKS
        mf = RKS(mol, xc="pbe0")
    else:
        mf = dft.RKS(mol, xc="pbe0")
    mf.grids.level = GRID_LEVEL; mf.grids.prune = dft.gen_grid.nwchem_prune        # same grid settings as the Skala runs
    mf.conv_tol = 1e-9; mf.max_memory = 14000
    mf = mf.density_fit(auxbasis="def2-universal-jkfit")                              # same density fitting as the Skala runs
    if d3bj: mf.disp = "d3bj"
    e = float(mf.kernel()); conv = bool(getattr(mf, "converged", True))
    del mf; hard_reset_gpu_allocator_pool()
    return e, conv
if RUN_PBE0:
    for shape in SHAPES:
        rdir = os.path.join(RES_ROOT, shape); f = os.path.join(rdir, "opt_final.xyz"); out = os.path.join(rdir, "pbe0_sp.json")
        if not os.path.exists(f): print(f"   {shape}: no optimized structure yet"); continue
        if os.path.exists(out): print(f"   [skip] {shape}: PBE0 single point already saved"); continue
        a = read(f); t0 = time.time(); rec = dict(shape=shape, basis=BASIS, geometry="Skala-1.1 optimized (opt_final.xyz)", functional="PBE0", grid_level=GRID_LEVEL, density_fitting="def2-universal-jkfit")
        e, c = pbe0_single_point(a); rec.update(E_Eh=e, converged=c)
        if PBE0_TRY_D3BJ:
            try:
                e2, c2 = pbe0_single_point(a, d3bj=True); rec.update(E_D3BJ_Eh=e2, converged_D3BJ=c2)
            except Exception as ex:
                rec.update(E_D3BJ_Eh=None, note_D3BJ=f"not available: {type(ex).__name__}: {str(ex)[:100]}")
        json.dump(rec, open(out, "w"), indent=2); print(f"   {shape}: PBE0/{BASIS} E = {e:.8f} Eh (converged {c}) in {time.time()-t0:.0f} s" + (f" | with D3(BJ): {rec.get('E_D3BJ_Eh')}" if PBE0_TRY_D3BJ else ""))


In [ ]:
# @title 10. IR spectrum from the saved data (CPU only: can be run alone later, even without a GPU)
import os, json, numpy as np
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from ase.io import read
from ase.vibrations import Infrared
try:
    RES_ROOT
except NameError:
    _rf = "fu5_results_run3" if globals().get("INPUT_SOURCE") == "deformed6" else "fu5_results"
    RES_ROOT = "/content/drive/MyDrive/" + _rf if os.path.isdir("/content/drive/MyDrive") else "/content/" + _rf
    try:
        from google.colab import drive; drive.mount("/content/drive"); RES_ROOT = "/content/drive/MyDrive/" + _rf
    except Exception: pass
IR_SHAPES = ["T1"]            # edit: ["T1", "T3", "T3B"]
FREQ_SCALE = 1.0              # harmonic scaling factor. 1.0 = unscaled. Set it from a source you can cite, never to match one band.
FWHM_CM1 = 10.0               # Lorentzian width of the broadened curve
for shape in IR_SHAPES:
    rdir = os.path.join(RES_ROOT, shape); cache = os.path.join(rdir, "ir_cache")
    if not (os.path.isdir(cache) and os.path.exists(os.path.join(rdir, "opt_final.xyz"))): print(f"{shape}: no saved data in {rdir}"); continue
    _n = len([f for f in os.listdir(cache) if f.endswith(".json") and os.path.getsize(os.path.join(cache, f)) > 0])
    if _n < 6 * 12 + 1: print(f"{shape}: the Hessian is not finished yet ({_n} of 73 calculations saved in {cache}). Run this cell again when the Hessian cell of that session is done."); continue
    atoms = read(os.path.join(rdir, "opt_final.xyz")); ir = Infrared(atoms, name=cache, delta=0.01, nfree=2)
    f = ir.get_frequencies(); I = np.array(ir.intensities) * ir.intensity_prefactor("km/mol")[0]
    keep = [k for k in range(len(f))][-(3 * len(atoms) - 6):]
    fr = np.array([np.real(f[k]) if np.imag(f[k]) == 0 else -np.imag(f[k]) for k in keep]) * FREQ_SCALE; it = I[keep]
    x = np.linspace(400, 4000, 3601); y = sum(a * (FWHM_CM1 / 2) ** 2 / ((x - c) ** 2 + (FWHM_CM1 / 2) ** 2) for c, a in zip(fr, it))
    np.savetxt(os.path.join(rdir, f"ir_curve_{shape}_scale{FREQ_SCALE}.csv"), np.c_[x, y], delimiter=",", header="wavenumber_cm1,intensity_km_per_mol (Lorentzian, FWHM %.1f)" % FWHM_CM1, comments="")
    fig, ax = plt.subplots(figsize=(9, 3.6)); ax.plot(x, y, lw=1.2); ax.vlines(fr, 0, it, color="#eb6834", lw=1)
    ax.set_xlim(4000, 400); ax.set_xlabel("wavenumber (cm$^{-1}$)"); ax.set_ylabel("intensity (km/mol)"); ax.set_title(f"{shape}: Skala-1.1 harmonic IR, scale {FREQ_SCALE}, gas phase")
    fig.tight_layout(); fig.savefig(os.path.join(rdir, f"ir_{shape}.png"), dpi=150); plt.show()
    print(f"{shape}: strongest bands (cm-1, km/mol):", [(round(float(a), 1), round(float(b), 1)) for a, b in sorted(zip(fr, it), key=lambda t: -t[1])[:6]])
print("Harmonic, unscaled unless FREQ_SCALE is set; double-harmonic intensities; gas phase. Not compared with any measured spectrum here.")


In [ ]:
# @title 11. T3 - T1 comparison (electronic and free energy) against the CBS-QB3 value
import os, json
try: RES_ROOT
except NameError:
    _rf = "fu5_results_run3" if globals().get("INPUT_SOURCE") == "deformed6" else "fu5_results"
    RES_ROOT = "/content/drive/MyDrive/" + _rf if os.path.isdir("/content/drive/MyDrive") else "/content/" + _rf
KC = 23.060548; CBS = 13.33
T = {}
for s in ("T1", "T3", "T3B"):
    p = os.path.join(RES_ROOT, s, "thermo.json")
    if os.path.exists(p): T[s] = json.load(open(p))
print("Available:", list(T))
if "T1" in T:
    print(f"{'quantity (relative to T1, kcal/mol)':42s} " + " ".join(f"{s:>9s}" for s in ("T3", "T3B") if s in T))
    rows = {"dE electronic": lambda t: (t["E_eV"] - T["T1"]["E_eV"]) * KC,
            "dE + ZPE": lambda t: (t["E_eV"] + t["ZPE_eV"] - T["T1"]["E_eV"] - T["T1"]["ZPE_eV"]) * KC,
            "dH(298 K)": lambda t: (t["H298_eV"] - T["T1"]["H298_eV"]) * KC,
            "dG(298 K)": lambda t: (t["G298_eV"] - T["T1"]["G298_eV"]) * KC}
    for name, fn in rows.items(): print(f"{name:42s} " + " ".join(f"{fn(T[s]):9.2f}" for s in ("T3", "T3B") if s in T))
    P = {}
    for s in ("T1", "T3", "T3B"):
        p = os.path.join(RES_ROOT, s, "pbe0_sp.json")
        if os.path.exists(p): P[s] = json.load(open(p))
    if "T1" in P:
        print(f"{'dE electronic, Skala (own geometries)':42s} " + " ".join(f"{(T[s]['E_eV'] - T['T1']['E_eV']) * KC:9.2f}" for s in ("T3", "T3B") if s in T))
        print(f"{'dE electronic, PBE0 at Skala geometries':42s} " + " ".join(f"{(P[s]['E_Eh'] - P['T1']['E_Eh']) * 627.5094740631:9.2f}" if s in P else f"{'-':>9s}" for s in ("T3", "T3B") if s in T))
        if all(P.get(s, {}).get("E_D3BJ_Eh") is not None for s in ("T1", "T3", "T3B") if s in T):
            print(f"{'dE electronic, PBE0-D3(BJ) at Skala geom.':42s} " + " ".join(f"{(P[s]['E_D3BJ_Eh'] - P['T1']['E_D3BJ_Eh']) * 627.5094740631:9.2f}" for s in ("T3", "T3B") if s in T))
        print("   The two electronic rows compare two functionals with each other. They say how far apart the functionals are, not which is closer to the truth.")
    cands = [s for s in ("T3", "T3B") if s in T]
    if cands:
        low = min(cands, key=lambda s: T[s]["G298_eV"]); dG = (T[low]["G298_eV"] - T["T1"]["G298_eV"]) * KC
        print(f"\nLower T3 rotamer by G(298): {low}. dG(T3 - T1) = {dG:.2f} kcal/mol; CBS-QB3 = {CBS} (accuracy about 1); deviation {dG - CBS:+.2f}; consistent within +/-1: {abs(dG - CBS) <= 1.0}")
    print("Imaginary modes:", {s: T[s]["n_imaginary"] for s in T}, "(zero expected)")
    print("Compare dG with dG only. The CBS-QB3 number is a free energy; do not set the electronic dE against it.")
else:
    print("Run the T1 shape first.")
